# FASSD Post-Processing Study (Round 2)

This notebook presents the evaluation of candidate post-processing strategies on the saved checkpoints from FASSD Run 1 (`fassd_gray_seed42` and `fassd_rgb_seed42`).

### Experimental Protocol
1. **Inference & Native Resolution**: Checkpoints are run on validation (254 images, 26 patients) and test (271 images, 25 patients) using the native $768 \times 1024$ resolution (resizing input to $384 \times 512$, forward pass, sigmoid probability output, bilinear interpolation back to $768 \times 1024$).
2. **Probability Caching**: Probability maps are cached as float16 `.npz` files outside the repository.
3. **Hyperparameter Tuning (VAL Only)**:
   - Threshold grid: $[0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]$
   - Morphology candidates: `none`, `largest` (keep largest connected component), and `min_area` with thresholds $\{50, 100, 200, 500, 1000, 2000\}$ pixels.
   - Selected strictly on validation set macro/structure Dice.
4. **Single-Pass Test Evaluation**: Selected settings are applied to the test set **once**.
5. **Statistical Rigor**:
   - Patient-level paired bootstrap (5,000 resamples, seed 42) for $\Delta \text{Dice}$ and $\Delta \text{HD95}$ with 95% confidence intervals.
   - Explicit accounting of empty ground-truth images and false-positive rates on empty slices.
   - Disambiguation of macro Dice definitions.


In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

RESULTS_JSON = Path("../../docs/results/fassd/postproc/fassd_postproc_results.json")
assert RESULTS_JSON.exists(), f"Results file not found: {RESULTS_JSON}"

with open(RESULTS_JSON, "r", encoding="utf-8") as f:
    results = json.load(f)

print(f"Models evaluated: {list(results.keys())}")


Models evaluated: ['fassd_gray_seed42', 'fassd_rgb_seed42']


## 1. Selected Post-Processing Configurations (Tuned on VAL)

In [2]:
config_rows = []
for model_name, m_data in results.items():
    sel = m_data["selected_config"]
    for struct, best in sel.items():
        config_rows.append({
            "Model": model_name,
            "Structure": struct,
            "Selected Threshold": best["threshold"],
            "Morphology Type": best["morph_type"],
            "Min Area (px)": best["min_area"],
            "VAL Dice": f"{best['val_dice']:.4f}",
            "VAL IoU": f"{best['val_iou']:.4f}",
        })

df_configs = pd.DataFrame(config_rows)
df_configs


,Model,Structure,Selected Threshold,Morphology Type,Min Area (px),VAL Dice,VAL IoU
0,fassd_gray_seed42,artery,0.60,largest,0,0.7091,0.6014
1,fassd_gray_seed42,liver,0.30,none,0,0.8563,0.7686
2,fassd_gray_seed42,stomach,0.35,min_area,200,0.7644,0.6626
3,fassd_gray_seed42,vein,0.50,largest,0,0.7996,0.6850
4,fassd_rgb_seed42,artery,0.30,largest,0,0.7080,0.5990
5,fassd_rgb_seed42,liver,0.35,largest,0,0.8570,0.7695
6,fassd_rgb_seed42,stomach,0.30,min_area,200,0.7634,0.6611
7,fassd_rgb_seed42,vein,0.40,largest,0,0.8011,0.6856


## 2. Test Performance: Baseline vs Post-Processed
The table below compares baseline ($T=0.5$, no morphology) against post-processed metrics on the test set (271 images, 25 patients), along with patient-level paired bootstrap 95% confidence intervals (5,000 resamples, seed 42).

In [3]:
test_rows = []
for model_name, m_data in results.items():
    test_data = m_data["test"]
    base_structs = test_data["baseline"]["by_structure"]
    post_structs = test_data["postproc"]["by_structure"]
    boot_structs = test_data["paired_diffs_5000boot_ci"]["by_structure"]
    
    for struct, base in base_structs.items():
        post = post_structs[struct]
        b_dice = boot_structs[struct]["dice"]
        b_hd = boot_structs[struct]["hd95"]
        
        test_rows.append({
            "Model": model_name,
            "Structure": struct,
            "Base Dice": f"{base['dice_mean']:.4f}",
            "Post Dice": f"{post['dice_mean']:.4f}",
            "Δ Dice (95% CI)": f"{b_dice['mean_diff']:+.4f} [{b_dice['ci_lower_95']:+.4f}, {b_dice['ci_upper_95']:+.4f}]",
            "Base HD95": f"{base['hd95_mean']:.2f}",
            "Post HD95": f"{post['hd95_mean']:.2f}",
            "Δ HD95 (95% CI)": f"{b_hd['mean_diff']:+.2f} [{b_hd['ci_lower_95']:+.2f}, {b_hd['ci_upper_95']:+.2f}]",
        })

df_test = pd.DataFrame(test_rows)
df_test


,Model,Structure,Base Dice,Post Dice,Δ Dice (95% CI),Base HD95,Post HD95,Δ HD95 (95% CI)
0,fassd_gray_seed42,artery,0.7592,0.7589,"+0.0004 [-0.0025, +0.0035]",10.41,9.35,"-0.93 [-2.40, -0.02]"
1,fassd_gray_seed42,liver,0.8777,0.8780,"-0.0000 [-0.0020, +0.0018]",24.59,24.82,"+0.25 [-0.13, +0.62]"
2,fassd_gray_seed42,stomach,0.7400,0.7413,"+0.0009 [-0.0008, +0.0027]",21.54,21.70,"+0.16 [-0.04, +0.42]"
3,fassd_gray_seed42,vein,0.7729,0.7714,"-0.0005 [-0.0041, +0.0031]",24.29,23.77,"-0.54 [-1.97, +0.94]"
4,fassd_rgb_seed42,artery,0.7645,0.7659,"+0.0008 [-0.0047, +0.0062]",13.95,13.69,"-0.26 [-0.70, +0.07]"
5,fassd_rgb_seed42,liver,0.8741,0.8748,"+0.0005 [-0.0028, +0.0039]",27.47,25.56,"-1.59 [-4.09, +0.17]"
6,fassd_rgb_seed42,stomach,0.7359,0.7386,"+0.0021 [-0.0004, +0.0048]",26.59,26.53,"-0.03 [-0.21, +0.18]"
7,fassd_rgb_seed42,vein,0.7717,0.7694,"-0.0021 [-0.0076, +0.0035]",20.04,19.70,"-0.32 [-2.85, +2.01]"


## 3. Empty Ground Truth Handling & False Positive Rates
For structures that are anatomically absent in certain scan slices ($|GT| = 0$), we report:
- Total number of empty GT slices in the test set.
- Detection count (slices where the model correctly predicts 0 pixels).
- Non-empty prediction rate on empty GT (slices with false positive predictions).

In [4]:
empty_rows = []
for model_name, m_data in results.items():
    test_data = m_data["test"]
    base_structs = test_data["baseline"]["by_structure"]
    post_structs = test_data["postproc"]["by_structure"]
    for struct, base in base_structs.items():
        post = post_structs[struct]
        empty_rows.append({
            "Model": model_name,
            "Structure": struct,
            "Empty GT Images": base["n_empty_gt"],
            "Base Correct Empty Rate": f"{base['empty_gt_detection_rate']*100:.1f}%",
            "Base Non-Empty Pred Rate": f"{base['non_empty_pred_rate_on_empty_gt']*100:.1f}%",
            "Post Correct Empty Rate": f"{post['empty_gt_detection_rate']*100:.1f}%",
            "Post Non-Empty Pred Rate": f"{post['non_empty_pred_rate_on_empty_gt']*100:.1f}%",
        })

df_empty = pd.DataFrame(empty_rows)
df_empty


,Model,Structure,Empty GT Images,Base Correct Empty Rate,Base Non-Empty Pred Rate,Post Correct Empty Rate,Post Non-Empty Pred Rate
0,fassd_gray_seed42,artery,1,0.0%,100.0%,0.0%,100.0%
1,fassd_gray_seed42,liver,0,100.0%,0.0%,100.0%,0.0%
2,fassd_gray_seed42,stomach,8,12.5%,87.5%,12.5%,87.5%
3,fassd_gray_seed42,vein,1,0.0%,100.0%,0.0%,100.0%
4,fassd_rgb_seed42,artery,1,0.0%,100.0%,0.0%,100.0%
5,fassd_rgb_seed42,liver,0,100.0%,0.0%,100.0%,0.0%
6,fassd_rgb_seed42,stomach,8,0.0%,100.0%,0.0%,100.0%
7,fassd_rgb_seed42,vein,1,0.0%,100.0%,0.0%,100.0%


## 4. Macro Dice Metric Disambiguation
We clarify the distinct formulations of Macro Dice:
1. **Per-Image Macro Dice**: Macro-average across non-empty structures present on each image.
2. **Image-Level Mean Macro Dice**: Mean of per-image macro Dice values across all test images.
3. **Structure-Level Macro Dice**: Unweighted arithmetic mean of individual structure-level Dice means.
4. **Patient-Level Macro Dice**: Macro-averaged across images belonging to each patient, providing the basis for patient-level paired bootstrap resampling.


In [5]:
print("=== SUMMARY METRICS ===")
for model_name, m_data in results.items():
    test_data = m_data["test"]
    b_eval = test_data["baseline"]
    p_eval = test_data["postproc"]
    m_dice_diff = test_data["paired_diffs_5000boot_ci"]["macro_dice"]
    m_hd_diff = test_data["paired_diffs_5000boot_ci"]["macro_hd95"]
    print(f"\n{model_name}:")
    print(f"  Structure-Mean Macro Dice:  Base = {b_eval['structure_macro_dice_mean']:.4f} -> Post = {p_eval['structure_macro_dice_mean']:.4f}")
    print(f"  Image-Level Macro Dice:     Base = {b_eval['image_level_macro_dice_mean']:.4f} -> Post = {p_eval['image_level_macro_dice_mean']:.4f}")
    print(f"  Patient-Level Macro Dice:   Base = {b_eval['patient_level_macro_dice_mean']:.4f} -> Post = {p_eval['patient_level_macro_dice_mean']:.4f}")
    print(f"  Paired Δ Dice (5k boot CI): {m_dice_diff['mean_diff']:+.4f} [{m_dice_diff['ci_lower_95']:+.4f}, {m_dice_diff['ci_upper_95']:+.4f}]")
    print(f"  Patient-Level Macro HD95:   Base = {b_eval['patient_level_macro_hd95_mean']:.2f} px -> Post = {p_eval['patient_level_macro_hd95_mean']:.2f} px")
    print(f"  Paired Δ HD95 (5k boot CI): {m_hd_diff['mean_diff']:+.2f} px [{m_hd_diff['ci_lower_95']:+.2f}, {m_hd_diff['ci_upper_95']:+.2f}]")


=== SUMMARY METRICS ===

fassd_gray_seed42:
  Structure-Mean Macro Dice:  Base = 0.7874 -> Post = 0.7874
  Image-Level Macro Dice:     Base = 0.7876 -> Post = 0.7876
  Patient-Level Macro Dice:   Base = 0.7933 -> Post = 0.7934
  Paired Δ Dice (5k boot CI): +0.0002 [-0.0010, +0.0013]
  Patient-Level Macro HD95:   Base = 19.37 px -> Post = 19.13 px
  Paired Δ HD95 (5k boot CI): -0.25 px [-0.86, +0.26]

fassd_rgb_seed42:
  Structure-Mean Macro Dice:  Base = 0.7866 -> Post = 0.7872
  Image-Level Macro Dice:     Base = 0.7866 -> Post = 0.7871
  Patient-Level Macro Dice:   Base = 0.7925 -> Post = 0.7927
  Paired Δ Dice (5k boot CI): +0.0002 [-0.0020, +0.0027]
  Patient-Level Macro HD95:   Base = 21.32 px -> Post = 20.79 px
  Paired Δ HD95 (5k boot CI): -0.54 px [-1.70, +0.32]
